# Project - Translator AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

/Users/srikanth/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialization

load_dotenv(override=True)

# openai_api_key = os.getenv('OPENAI_API_KEY')
# if openai_api_key:
#     print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
# else:
#     print("OpenAI API Key not set")
    
# MODEL = "gpt-4.1-mini"
MODEL = "llama3.2"
openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

DB = "translated.db"

In [3]:
system_message = """
You are a helpful assistant to teach a language to customers. 
You have to translate the english typed language to telugu language. 
Customer may also ask in english audio also, then we need to convert it to text and translate. 
"""

In [6]:
def get_translated_text(english_text):
    print(f"DATABASE TOOL CALLED: Getting translated text for {english_text}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT telugu_text FROM translator WHERE english_text = ?', (english_text.lower(),))
        result = cursor.fetchone()
        return f"Translated from {english_text} to ${result[0]}"

In [7]:
get_translated_text("How are you")

DATABASE TOOL CALLED: Getting translated text for How are you


OperationalError: no such table: translator

In [14]:
translated_function = {
    "name": "get_translated_text",
    "description": "Get the translated text to telugu for the given english sentence.",
    "parameters": {
        "type": "object",
        "properties": {
            "english_text": {
                "type": "string",
                "description": "The english sentence to be translated to telugu",
            },
        },
        "required": ["english_text"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": translated_function}]
tools


[{'type': 'function',
  'function': {'name': 'get_translated_text',
   'description': 'Get the translated text to telugu for the given english sentence.',
   'parameters': {'type': 'object',
    'properties': {'english_text': {'type': 'string',
      'description': 'The english sentence to be translated to telugu'}},
    'required': ['english_text'],
    'additionalProperties': False}}}]

In [15]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [19]:
import sqlite3

DB = "translated.db"

# Connect to the database and create the table
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS translator (
            english_text TEXT,
            telugu_text TEXT
        )
    ''')
    
    # Insert some sample data (notice we use lowercase for english_text 
    # to match your .lower() in the query)
    sample_data = [
        ("how are you", "మీరు ఎలా ఉన్నారు"),
        ("hello", "నమస్కారం"),
        ("thank you", "ధన్యవాదాలు"),
        ("good morning", "శుభోదయం")
    ]
    
    # We use INSERT OR IGNORE or just standard insert if empty. 
    # Let's clear the table first just to be safe if you run this multiple times
    cursor.execute('DELETE FROM translator')
    cursor.executemany('''
        INSERT INTO translator (english_text, telugu_text) 
        VALUES (?, ?)
    ''', sample_data)
    
    conn.commit()

print("Database initialized successfully!")


Database initialized successfully!


In [20]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, 
        messages=messages, 
        tools=tools)
    
    return response.choices[0].message.content

In [21]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_translated_text":
            arguments = json.loads(tool_call.function.arguments)
            english_text = arguments.get('english_text')
            telugu_text = get_translated_text(english_text)
            responses.append({
                "role": "tool",
                "content": telugu_text,
                "tool_call_id": tool_call.id
            })
    return responses

In [22]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting translated text for Good morning
DATABASE TOOL CALLED: Getting translated text for Good morning
DATABASE TOOL CALLED: Getting translated text for How are you
DATABASE TOOL CALLED: Getting translated text for how are you
DATABASE TOOL CALLED: Getting translated text for good morning
DATABASE TOOL CALLED: Getting translated text for why i am see different language other than telugu


Traceback (most recent call last):
  File "/Users/srikanth/miniconda3/lib/python3.13/site-packages/gradio/queueing.py", line 882, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "/Users/srikanth/miniconda3/lib/python3.13/site-packages/gradio/route_utils.py", line 417, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<12 lines>...
    )
    ^
  File "/Users/srikanth/miniconda3/lib/python3.13/site-packages/gradio/blocks.py", line 2430, in process_api
    result = await self.call_function(
             ^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<9 lines>...
    )
    ^
  File "/Users/srikanth/miniconda3/lib/python3.13/site-packages/gradio/blocks.py", line 1696, in call_function
    prediction = await fn(*processed_input)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/srikanth/miniconda3/lib/python3.13/s

# Let's go multi-modal!!

We can use gpt-image-1-mini, the image generation model behind GPT-5, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 3 cents - don't go crazy with images!

In [31]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image
import time

from transformers import pipeline

In [32]:
transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-tiny.en")

Loading weights: 100%|██████████| 167/167 [00:00<00:00, 1082.62it/s, Materializing param=model.encoder.layers.3.self_attn_layer_norm.weight]  


In [33]:
def transcribe_audio(audio_filepath):
    """Converts the recorded audio file to text."""
    if not audio_filepath:
        return ""
        
    #--- Integration with OpenAI Whisper ---
    result = transcriber(audio_filepath)
    print(f"Transcribed Text: {result['text']}")
    return result['text']
    
    # Fallback/mock for testing without an API call
    print(f"Received audio file at: {audio_filepath}")
    return "how are you" # Hardcoded for testing. Uncomment Whisper above for real use.

def process_and_stream(audio_filepath):
    """Transcribes audio, queries the DB, and streams the result."""
    if not audio_filepath:
        yield "Please record an audio first."
        return
        
    # 1. Transcribe the audio
    english_text = transcribe_audio(audio_filepath)
    
    # 2. Look up the database
    translated_text = None
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        # Clean the transcription to maximize match probability
        clean_english = english_text.lower().strip().strip('.,!?')
        
        cursor.execute('SELECT telugu_text FROM translator WHERE english_text = ?', (clean_english,))
        result = cursor.fetchone()
        
        if result:
            translated_text = result[0]
        else:
            translated_text = f"Sorry, no translation found in DB for: '{english_text}'"
    # 3. Stream the translated text
    # Gradio automatically streams the output if your function is a generator (uses yield)
    streamed_response = ""
    for char in translated_text:
        streamed_response += char
        yield streamed_response
        time.sleep(0.05) # Small delay to simulate the streaming effect


In [37]:
# 4. Build the Gradio UI
with gr.Blocks(title="Voice to Telugu DB Translator") as voice_ui:
    gr.Markdown("## Voice to Telugu Translator")
    gr.Markdown("Speak an English phrase that exists in the database (e.g., 'how are you', 'hello').")
    
    with gr.Row():
        audio_input = gr.Audio(sources=["microphone"], type="filepath", label="Record English Speech")
        text_output = gr.Textbox(label="Telugu Translation", interactive=False)
        
    submit_btn = gr.Button("Translate", variant="primary")
    
    # Link the button click to our streaming generator function
    submit_btn.click(
        fn=process_and_stream,
        inputs=audio_input,
        outputs=text_output
    )
# Launch the app
voice_ui.launch()

* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
